# Módulo 14 · Python aplicado a SIG y proyecto integrador

**Curso:** Python desde cero · **Duración estimada:** 4 h

## Objetivos

Al terminar este módulo usted podrá:

- Diferenciar coordenadas geográficas y UTM.
- Convertir coordenadas a UTM WGS84 Zona 17S (EPSG:32717) con pyproj.
- Calcular distancias, perímetros y áreas a partir de coordenadas UTM.
- Integrar lo aprendido en un pequeño procesador de inventario.

## Cómo usar este cuaderno

1. Lea cada lección y ejecute los ejemplos con **Shift + Enter**.
2. Modifique los ejemplos: cambie valores y observe qué pasa. Así se aprende.
3. Resuelva los ejercicios en la celda indicada y ejecute la **celda de comprobación**.
4. Si se atasca, abra la **pista**. Las soluciones están en la carpeta `soluciones/`.
5. Termine con la **autoevaluación**.

## 1. Coordenadas geográficas y UTM

- **Geográficas (EPSG:4326, WGS84):** latitud y longitud en grados. En el Perú la latitud es negativa (hemisferio sur) y la longitud también (oeste).
- **UTM (EPSG:32717, WGS84 Zona 17S):** Este y Norte en **metros**. Permite medir distancias y áreas directamente con geometría plana.

Para calcular distancias o áreas, primero convierta a UTM. El orden de los ejes importa: con `always_xy=True`, pyproj recibe **(longitud, latitud)** y devuelve **(Este, Norte)**.

## 2. Convertir coordenadas con pyproj

`pyproj` es el paquete estándar para transformar coordenadas (lo usan QGIS y geopandas internamente). Requiere instalación (`%pip install pyproj`), por lo que este ejemplo **solo se ejecuta en Jupyter o Colab**.

*Los puntos son de ejemplo.*

In [ ]:
from pyproj import Transformer

a_utm = Transformer.from_crs("EPSG:4326", "EPSG:32717", always_xy=True)

puntos = {"Punto 1": (-80.63, -5.20), "Punto 2": (-79.97, -5.09)}  # (lon, lat)
for nombre, (lon, lat) in puntos.items():
    este, norte = a_utm.transform(lon, lat)
    print(f"{nombre}: E = {este:,.1f} m   N = {norte:,.1f} m")

# Conversión inversa: UTM -> geográficas
a_geo = Transformer.from_crs("EPSG:32717", "EPSG:4326", always_xy=True)
lon, lat = a_geo.transform(540000, 9425000)
print(f"lon = {lon:.6f}, lat = {lat:.6f}")

Punto 1: E = 541,003.7 m   N = 9,425,215.9 m
Punto 2: E = 614,170.3 m   N = 9,437,296.2 m
lon = -80.639056, lat = -5.201959


## 3. Distancia y perímetro en UTM

En UTM la distancia entre dos puntos es la **distancia euclidiana** (Pitágoras). `math.dist(p1, p2)` la calcula directamente. El perímetro de un polígono es la suma de las distancias entre vértices consecutivos, cerrando con el primero.

In [ ]:
import math

vertices = [(540000, 9425000), (540200, 9425000), (540200, 9425150), (540000, 9425150)]

perimetro = 0
for i in range(len(vertices)):
    p1 = vertices[i]
    p2 = vertices[(i + 1) % len(vertices)]   # el último se une con el primero
    perimetro += math.dist(p1, p2)
print(f"Perímetro: {perimetro:,.1f} m")

Perímetro: 700.0 m


## 4. Área de un polígono (fórmula del zapatero)

Con los vértices en UTM, el área de un polígono simple se obtiene con la **fórmula del zapatero (shoelace)**:

Área = ½ · |Σ (xᵢ · yᵢ₊₁ − xᵢ₊₁ · yᵢ)|

Los vértices deben estar **en orden** (horario o antihorario) y el polígono no debe cruzarse a sí mismo.

In [ ]:
def area_poligono(vertices):
    """Área (m²) de un polígono simple con vértices UTM ordenados."""
    n = len(vertices)
    suma = 0
    for i in range(n):
        x1, y1 = vertices[i]
        x2, y2 = vertices[(i + 1) % n]
        suma += x1 * y2 - x2 * y1
    return abs(suma) / 2

lote = [(540000, 9425000), (540200, 9425000), (540200, 9425150), (540000, 9425150)]
a = area_poligono(lote)
print(f"Área: {a:,.0f} m2 = {a / 10000:.2f} ha")

Área: 30,000 m2 = 3.00 ha


## 5. Ecosistema de Python para SIG

| Paquete | Uso | Ejemplo de tarea |
|---|---|---|
| **pyproj** | Sistemas de referencia | Convertir lat/lon a UTM 17S |
| **shapely** | Geometrías | Áreas, buffers, intersecciones |
| **geopandas** | Capas vectoriales como tablas | Leer shapefile/GeoPackage, `to_crs(32717)` |
| **rasterio** | Rásteres | Leer un DEM o un índice (NDVI, MSAVI) |
| **PyQGIS** | Automatizar QGIS | Procesos por lotes en la consola de QGIS |
| **openpyxl** | Excel | Reportes con formato |

Ruta sugerida después de este curso: geopandas → rasterio → PyQGIS.

## 6. Proyecto integrador: procesador de inventario

Este programa reúne lo aprendido: **lectura de CSV** (M11), **funciones** (M09), **diccionarios** (M08), **control de errores** (M10) y **f-strings** (M04). Lee un inventario, valida los datos, calcula área basal y volumen y emite un resumen por especie.

**Reto final:** adáptelo para (1) leer un archivo real con `open()`, (2) exportar el resumen a CSV, y (3) rehacerlo con pandas (M13).

*Datos ficticios; ff = 0.5 supuesto.*

In [ ]:
import csv
import io
import math

DATOS = """codigo,especie,dap_cm,altura_m
A-01,Algarrobo,32.5,9.0
A-02,Faique,18.0,6.5
A-03,Algarrobo,45.2,11.0
A-04,Hualtaco,22.7,8.2
A-05,Faique,abc,7.0
A-06,Algarrobo,38.1,10.4
"""
FF = 0.5   # factor de forma supuesto

def area_basal(dap_cm):
    return math.pi / 4 * (dap_cm / 100) ** 2

def procesar(texto_csv):
    resumen = {}
    observados = []
    for fila in csv.DictReader(io.StringIO(texto_csv)):
        try:
            dap = float(fila["dap_cm"])
            alt = float(fila["altura_m"])
        except ValueError:
            observados.append(fila["codigo"])
            continue
        ab = area_basal(dap)
        r = resumen.setdefault(fila["especie"], {"n": 0, "ab": 0.0, "vol": 0.0})
        r["n"] += 1
        r["ab"] += ab
        r["vol"] += ab * alt * FF
    return resumen, observados

resumen, observados = procesar(DATOS)
print(f"{'Especie':<12}{'N':>4}{'AB (m2)':>10}{'Vol (m3)':>10}")
for especie, r in sorted(resumen.items()):
    print(f"{especie:<12}{r['n']:>4}{r['ab']:>10.4f}{r['vol']:>10.4f}")
print("Registros por verificar:", observados)

Especie        N   AB (m2)  Vol (m3)
Algarrobo      3    0.3574    1.8487
Faique         1    0.0254    0.0827
Hualtaco       1    0.0405    0.1659
Registros por verificar: ['A-05']


---
# Ejercicios

Escriba su solución en la celda de código de cada ejercicio, ejecútela y luego ejecute la **celda de comprobación** que está debajo. Si todo está bien verá `✔ ¡Correcto!`; si no, un mensaje le indicará qué revisar.

Niveles: ★☆☆ básico · ★★☆ intermedio · ★★★ reto.

### Ejercicio 1 · Función distancia  ★☆☆

Complete `distancia(p1, p2)` que devuelva la distancia en metros entre dos puntos UTM (tuplas).

<details><summary><b>Ver pista</b></summary>

Desempaquete: `x1, y1 = p1`, y aplique Pitágoras; o use `math.dist(p1, p2)`.

</details>

In [ ]:
def distancia(p1, p2):
    pass

print(distancia((540000, 9425000), (540300, 9425400)))

In [ ]:
# Comprobación del ejercicio (no modifique esta celda)
assert abs(distancia((540000, 9425000), (540300, 9425400)) - 500) < 1e-9, "Debe ser 500 m."
assert distancia((0, 0), (0, 0)) == 0, "La distancia de un punto a sí mismo es 0."
print("✔ ¡Correcto! Ejercicio resuelto.")

### Ejercicio 2 · Área en hectáreas  ★★☆

Complete `area_ha(vertices)` que devuelva el área del polígono en **hectáreas** con la fórmula del zapatero.

<details><summary><b>Ver pista</b></summary>

Reutilice el ciclo de la lección y divida el resultado final entre 10 000.

</details>

In [ ]:
def area_ha(vertices):
    pass

triangulo = [(540000, 9425000), (540400, 9425000), (540000, 9425300)]
print(area_ha(triangulo))

In [ ]:
# Comprobación del ejercicio (no modifique esta celda)
tri = [(540000, 9425000), (540400, 9425000), (540000, 9425300)]
assert abs(area_ha(tri) - 6.0) < 1e-9, "El triángulo de 400 x 300 m tiene 6 ha."
cuadrado = [(0, 0), (100, 0), (100, 100), (0, 100)]
assert abs(area_ha(cuadrado) - 1.0) < 1e-9, "Un cuadrado de 100 m de lado tiene 1 ha."
assert abs(area_ha(list(reversed(cuadrado))) - 1.0) < 1e-9, "El sentido de los vértices no debe cambiar el área."
print("✔ ¡Correcto! Ejercicio resuelto.")

### Ejercicio 3 · Perímetro  ★★☆

Complete `perimetro(vertices)` que devuelva el perímetro en metros de un polígono cerrado.

<details><summary><b>Ver pista</b></summary>

Sume `math.dist(vertices[i], vertices[(i + 1) % n])` para cada i.

</details>

In [ ]:
import math

def perimetro(vertices):
    pass

print(perimetro([(0, 0), (300, 0), (300, 400)]))

In [ ]:
# Comprobación del ejercicio (no modifique esta celda)
assert abs(perimetro([(0, 0), (300, 0), (300, 400)]) - 1200) < 1e-9, "300 + 400 + 500 = 1200 m."
print("✔ ¡Correcto! Ejercicio resuelto.")

### Ejercicio 4 · Centro aproximado  ★★☆

Complete `promedio_vertices(vertices)` que devuelva la tupla (Este, Norte) promedio de los vértices. *Nota: en polígonos irregulares no es el centroide geométrico exacto.*

<details><summary><b>Ver pista</b></summary>

Use comprensiones: `sum(x for x, y in vertices) / len(vertices)` y lo mismo para y.

</details>

In [ ]:
def promedio_vertices(vertices):
    pass

print(promedio_vertices([(0, 0), (200, 0), (200, 100), (0, 100)]))

In [ ]:
# Comprobación del ejercicio (no modifique esta celda)
assert promedio_vertices([(0, 0), (200, 0), (200, 100), (0, 100)]) == (100, 50), "Se esperaba (100, 50)."
print("✔ ¡Correcto! Ejercicio resuelto.")

---
# Autoevaluación

Responda mentalmente y luego despliegue la respuesta.

**1. ¿Qué código EPSG corresponde a UTM WGS84 Zona 17S?**

   a) 4326
   b) 32717
   c) 32617
   d) 3857

<details><summary>Ver respuesta</summary>

**b)** 32717. EPSG:32717 = WGS84 / UTM zona 17S (326xx es hemisferio norte).

</details>

**2. ¿Por qué conviene convertir a UTM antes de medir áreas?**

   a) Porque UTM usa metros y permite geometría plana
   b) Porque ocupa menos memoria
   c) Porque lo exige Python
   d) No conviene

<details><summary>Ver respuesta</summary>

**a)** Porque UTM usa metros y permite geometría plana. Las coordenadas UTM están en metros; los grados no son unidades de longitud constantes.

</details>

**3. Con `always_xy=True`, ¿en qué orden recibe pyproj las coordenadas geográficas?**

   a) (latitud, longitud)
   b) (longitud, latitud)
   c) (norte, este)
   d) Indistinto

<details><summary>Ver respuesta</summary>

**b)** (longitud, latitud). always_xy fuerza el orden x, y: longitud, latitud.

</details>

**4. ¿Qué requisito tiene la fórmula del zapatero?**

   a) Vértices ordenados y polígono sin cruces
   b) Solo triángulos
   c) Coordenadas en grados
   d) Exactamente 4 vértices

<details><summary>Ver respuesta</summary>

**a)** Vértices ordenados y polígono sin cruces. Funciona para polígonos simples con vértices en orden.

</details>

# Glosario del módulo

| Término | Significado |
|---|---|
| **EPSG** | Código que identifica un sistema de referencia de coordenadas. |
| **UTM** | Proyección que expresa coordenadas en metros (Este, Norte) por zonas. |
| **pyproj** | Paquete de Python para transformar coordenadas. |
| **Fórmula del zapatero** | Método para calcular el área de un polígono a partir de sus vértices. |
| **geopandas** | Extensión de pandas para capas geográficas vectoriales. |

# Antes de continuar

Marque lo que ya domina:

- [ ] Diferenciar coordenadas geográficas y UTM.
- [ ] Convertir coordenadas a UTM WGS84 Zona 17S (EPSG:32717) con pyproj.
- [ ] Calcular distancias, perímetros y áreas a partir de coordenadas UTM.
- [ ] Integrar lo aprendido en un pequeño procesador de inventario.